# 03 — Retrieval evaluation

Загружаем manifest из ноутбука 02 (список коллекций и датасетов),
загружаем QA-вопросы для каждого датасета (синтетические + native ground-truth),
и прогоняем retrieval grid: для каждой (collection, question) пары
вычисляем recall@k, precision@k, MRR, nDCG@k.

Поддерживает два формата ground truth:
- **span** (synthetic BPMSoft, HotpotQA): char-offset спаны, оценка по overlap
- **document** (BEIR): document-level relevance, оценка по presence в top-k

In [ ]:
import sys
import json
import time

sys.path.insert(0, "..")

from config import (
    TOP_K,
    K_SET,
    RELEVANCE_THRESHOLD,
    RELEVANCE_METRIC,
    DATA_DIR,
    EVAL_DATASETS,
)

In [ ]:
with open(DATA_DIR / "eval_manifest.json", encoding="utf-8") as f:
    manifest = json.load(f)

collection_names = manifest["collections"]
dataset_keys = manifest["datasets"]
print(f"Manifest: {len(dataset_keys)} datasets, {len(collection_names)} collections")
print(f"Datasets: {dataset_keys}")
print(f"Embedding: {manifest['embedding']['model_name']}")
print(f"Chunking: {manifest['chunking']}")

## 1. Загрузка QA-вопросов

- **synthetic**: из `synthetic_qa.jsonl` (ноутбук 01)
- **BEIR / HotpotQA**: native ground-truth QA (без генерации)

In [ ]:
from evalkit.dataset_loaders import (
    load_ir_dataset,
    load_hotpotqa,
    load_synthetic,
    load_rubq_retrieval,
)

all_qa = []

# Synthetic BPMSoft
synthetic_cfg = EVAL_DATASETS.get("synthetic_bpmsoft")
if synthetic_cfg and synthetic_cfg["type"] == "synthetic":
    print("Loading synthetic BPMSoft QA...")
    _, qa_rows = load_synthetic(DATA_DIR / synthetic_cfg["path"])
    all_qa.extend(qa_rows)
    print(f"  {len(qa_rows)} questions")

# BEIR / HotpotQA / RuBQ datasets
for ds_key in dataset_keys:
    ds_cfg = EVAL_DATASETS.get(ds_key)
    if not ds_cfg:
        continue
    if ds_cfg["type"] == "beir":
        print(f"Loading BEIR: {ds_key}...")
        _, qa_rows = load_ir_dataset(hf_name=ds_cfg["hf_name"], dataset_source=ds_key)
        all_qa.extend(qa_rows)
        print(f"  {len(qa_rows)} questions")
    elif ds_cfg["type"] == "hotpotqa":
        print(f"Loading HotpotQA: {ds_key}...")
        _, qa_rows = load_hotpotqa(
            hf_name=ds_cfg["hf_name"],
            max_examples=ds_cfg.get("max_examples"),
            dataset_source=ds_key,
        )
        all_qa.extend(qa_rows)
        print(f"  {len(qa_rows)} questions")
    elif ds_cfg["type"] == "rubq":
        print(f"Loading RuBQ Retrieval: {ds_key}...")
        _, qa_rows = load_rubq_retrieval(
            hf_name=ds_cfg["hf_name"], dataset_source=ds_key
        )
        all_qa.extend(qa_rows)
        print(f"  {len(qa_rows)} questions")

qa_answerable = [q for q in all_qa if q.get("gold_spans")]
qa_unanswerable = [q for q in all_qa if not q.get("gold_spans")]
sources = sorted({q.get("dataset_source", "unknown") for q in all_qa})
print(f"\nTotal: {len(all_qa)} questions")
print(f"Sources: {sources}")
print(f"Answerable: {len(qa_answerable)}, Unanswerable: {len(qa_unanswerable)}")

## 2. Retrieval grid: collection × dataset

In [ ]:
from evalkit.client import make_client
from evalkit.eval_runner import run_retrieval_grid

client = make_client()
dataset_collections = {
    cd["dataset_key"]: cd["collection_name"] for cd in manifest["collection_details"]
}
print(f"Dataset-to-collection mapping: {dataset_collections}")
print(f"top_k={TOP_K}, k_set={K_SET}, threshold={RELEVANCE_THRESHOLD}")
print()
t0 = time.perf_counter()
results_df = run_retrieval_grid(
    qa_dataset=qa_answerable,
    client=client,
    top_k=TOP_K,
    k_set=K_SET,
    dataset_collections=dataset_collections,
    relevance_threshold=RELEVANCE_THRESHOLD,
    relevance_metric=RELEVANCE_METRIC,
)
elapsed = time.perf_counter() - t0
out_path = DATA_DIR / "retrieval_results.csv"
results_df.to_csv(out_path, index=False)
print(f"\nSaved {len(results_df)} rows -> {out_path}")
results_df.head()

## 3. Агрегация по collection × dataset

In [ ]:
pivot = results_df.groupby(["dataset_source", "collection"])[
    ["recall@1", "recall@3", "recall@5", "precision@5", "mrr", "ndcg@5"]
].mean()
print("Mean metrics per collection × dataset:")
pivot.round(3)